# Pixel Relational Motif — E0.1 Kaggle Run

Source-locked execution notebook for GitHub Issue #74. This notebook runs **E0.1 only**: Train-only motif dictionary/stability/non-degeneracy followed by Train-only occurrence calibration. PublicTest is not required and PrivateTest must not be read.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Irthn1311/FER2013_Graph.git"
REPO_BRANCH = "research/pixel-relational-motif-e0"
SOURCE_SHA = "5fda000413c4dcfe810917f07e37bcafb1c8d394"
PACKAGE_RELATIVE = Path("research/pixel_relational_motif_e0")
FER_INPUT_ROOT = Path("/kaggle/input/datasets/doduyquynii/fer13-split/fer13-split")
TRAIN_CSV = FER_INPUT_ROOT / "train.csv"
OUTPUT_DIR = Path("/kaggle/working/outputs/pixel_relational_motif_e0/e01")
RUN_TESTS = True
RUN_E01 = True
RUN_OCCURRENCE = True

print("PGM E0.1 configuration")
print("  source SHA:", SOURCE_SHA)
print("  Train only:", TRAIN_CSV)
print("  output:", OUTPUT_DIR)


## Exact source checkout and import isolation

In [ ]:
import os
import shutil
import subprocess
import sys

WORKING = Path("/kaggle/working")
PROJECT = WORKING / "FER2013_Graph_E01"
if PROJECT.exists():
    shutil.rmtree(PROJECT)
subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", SOURCE_SHA], check=True)
head = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
if head != SOURCE_SHA:
    raise RuntimeError(f"source lock mismatch: {head} != {SOURCE_SHA}")
subprocess.run(["git", "-C", str(PROJECT), "diff", "--quiet"], check=True)
subprocess.run(["git", "-C", str(PROJECT), "diff", "--cached", "--quiet"], check=True)
PACKAGE = PROJECT / PACKAGE_RELATIVE
PACKAGE_SRC = PACKAGE / "src"
if not PACKAGE_SRC.is_dir():
    raise FileNotFoundError(PACKAGE_SRC)
sys.path.insert(0, str(PACKAGE_SRC))
import pixel_relational_motif_e0
imported = Path(pixel_relational_motif_e0.__file__).resolve()
if PACKAGE_SRC.resolve() not in imported.parents:
    raise RuntimeError(f"import isolation violation: {imported}")
print("Source lock PASS:", head)
print("Import isolation PASS:", imported)


## Environment and bounded package tests

In [ ]:
import json
import platform
import numpy as np
import scipy
import sklearn

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "source_sha": SOURCE_SHA,
}
print(json.dumps(environment, indent=2))
(OUTPUT_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
if RUN_TESTS:
    result = subprocess.run([sys.executable, "-m", "pytest", str(PACKAGE / "tests"), "-q"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"E0 package tests failed with exit code {result.returncode}")


## Train-only data gate

In [ ]:
if not TRAIN_CSV.is_file():
    raise FileNotFoundError(f"official Train CSV not found: {TRAIN_CSV}")
# No PublicTest/val path and no PrivateTest path are configured in this notebook.
print("Train input present. No dev/test input configured.")


## Long-run heartbeat

In [ ]:
import contextlib
import threading
import time

@contextlib.contextmanager
def heartbeat(label, interval_seconds=300):
    stop = threading.Event()
    started = time.time()
    def worker():
        while not stop.wait(interval_seconds):
            elapsed = (time.time() - started) / 60.0
            print(f"[heartbeat] {label}: {elapsed:.1f} min elapsed", flush=True)
    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    try:
        yield
    finally:
        stop.set()
        thread.join(timeout=1)
        print(f"[heartbeat] {label}: complete after {(time.time()-started)/60.0:.1f} min", flush=True)


## E0.1 dictionary, K-selection, stability and non-degeneracy

In [ ]:
from pixel_relational_motif_e0.e01_runner import run_e01

if RUN_E01:
    with heartbeat("E0.1 dictionary/stability"):
        e01_summary = run_e01(TRAIN_CSV, OUTPUT_DIR)
    print(json.dumps({
        "selected_k": e01_summary["k_selection"]["selected_k"],
        "one_se_candidates": e01_summary["k_selection"]["one_se_candidates"],
        "stable_component_count": e01_summary["canonical_dictionary"]["stable_component_count"],
        "private_test_read": e01_summary["private_test_read"],
        "public_test_read": e01_summary["public_test_read"],
    }, indent=2))


## E0.1 occurrence calibration

In [ ]:
from pixel_relational_motif_e0.e01_occurrence_runner import run_occurrence_calibration

dictionary_npz = OUTPUT_DIR / "e01_dictionary.npz"
if RUN_OCCURRENCE:
    if not dictionary_npz.is_file():
        raise FileNotFoundError(dictionary_npz)
    with heartbeat("E0.1 occurrence calibration"):
        occurrence_summary = run_occurrence_calibration(TRAIN_CSV, dictionary_npz, OUTPUT_DIR)
    print(json.dumps(occurrence_summary, indent=2))


## Final artifact inventory

This cell inventories artifacts only. It does not interpret E0.1 as scientifically positive or negative.

In [ ]:
artifacts = sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_file())
print("Artifacts:")
for name in artifacts:
    print(" -", name)
required = {"environment.json", "e01_summary.json", "e01_dictionary.npz", "e01_occurrence_summary.json", "e01_occurrence_counts.npz"}
missing = sorted(required - set(artifacts))
if missing:
    raise RuntimeError(f"missing required E0.1 artifacts: {missing}")
print("E0.1 execution artifacts complete. Scientific interpretation remains separate.")
